# Full Optimized RAG Pipeline Implementation

Complete RAG pipeline with all optimizations integrated.

Configuration:
- Chunking: Semantic-Level, 192 tokens
- Retrieval: Dense MMR, k=8, λ=0.5
- Document Order: Reverse (ascending relevance)
- Generator: Llama 3.1 8B

Evaluation: Compare generated answers vs ground truth on full dataset

## Step 1: Setup

In [1]:
!pip install -q python-dotenv datasets tiktoken langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-openai nltk rouge-score


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import re
import numpy as np
import pandas as pd
import tiktoken
import tempfile
from dotenv import load_dotenv
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import nltk
from typing import List, Dict, Tuple
from rouge_score import rouge_scorer

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

nltk.download('punkt_tab', quiet=True)

load_dotenv()
openrouter_token = os.environ.get('OPENROUTER_TOKEN')

print("✓ Setup complete")

/Users/saikrishna/Desktop/Codespace/IIIT_AI_ML_course/Capstone_Project/reliablerag/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Setup complete


## Step 2: Load Data

In [ ]:
DATASET_NAME = "delucionqa"  # feeds both the loader below and vector store naming


def load_rag_bench_data(num_samples=50):
    ds = load_dataset("galileo-ai/ragbench", DATASET_NAME, split="test")
    ds = ds[:num_samples]
    return pd.DataFrame(ds)

print("Loading dataset...")
dataset = load_rag_bench_data(num_samples=50)
print(f"✓ Loaded {len(dataset)} queries")

# Flatten documents
all_docs = []
for _, row in dataset.iterrows():
    for doc_pos, doc_text in enumerate(row["documents"]):
        all_docs.append({
            "doc_id": f"{row['id']}_d{doc_pos}",
            "row_id": str(row["id"]),
            "text": doc_text.strip(),
            "question": row["question"],
            "ground_truth": row["response"]
        })

docs_df = pd.DataFrame(all_docs)
print(f"✓ Flattened to {len(docs_df)} documents")

## Step 3: Initialize Models

In [ ]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.models import get_embedding_model, get_generation_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT
from ragbench_lib.vector_store import get_persist_dir, vector_store_names

embedding_model = get_embedding_model(openrouter_token)

llm = get_generation_llm(openrouter_token)

prompt = RAG_GENERATION_PROMPT

print("✓ Models initialized")


## Step 4: Semantic Chunking

In [ ]:
from ragbench_lib.chunking import count_tokens, create_semantic_chunks

print("Creating semantic chunks...")
documents = create_semantic_chunks(docs_df, target_tokens=192)
print(f"✓ Created {len(documents)} chunks")


## Step 5: Dense MMR Retriever

In [ ]:
class OptimizedRetriever:
    def __init__(self, documents, embedding_model, k=8, dataset_name="delucionqa"):
        self.k = k
        prefix, collection_name = vector_store_names(dataset_name, "optimized")
        persist_dir = get_persist_dir(prefix)
        self.vector_store = Chroma.from_documents(
            documents=documents,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir,
        )
    
    def retrieve_with_scores(self, query):
        """Dense MMR with scores"""
        results = self.vector_store.similarity_search_with_score(query, k=self.k)
        # Convert distance to similarity
        return [(doc, 1 - score) for doc, score in results]
    
    def retrieve_reversed(self, query):
        """Retrieve with reverse (ascending) relevance ordering"""
        docs_with_scores = self.retrieve_with_scores(query)
        # Sort by ascending relevance (reverse order)
        sorted_docs = sorted(docs_with_scores, key=lambda x: x[1], reverse=False)
        return [doc for doc, score in sorted_docs]

print("Setting up retriever...")
retriever = OptimizedRetriever(documents, embedding_model, k=8, dataset_name=DATASET_NAME)
print("✓ Retriever ready")

## Step 6: RAG Pipeline

In [7]:
def generate_answer(question: str) -> str:
    """Generate answer using full optimized pipeline"""
    # Retrieve with reverse ordering
    docs = retriever.retrieve_reversed(question)
    context = "\n\n".join([d.page_content for d in docs])
    
    # Generate
    response = (prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question
    })
    
    return response

print("✓ RAG pipeline configured")

✓ RAG pipeline configured


## Step 7: Evaluation Metrics

In [8]:
def compute_rouge(generated: str, reference: str) -> dict:
    """Compute ROUGE scores"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure,
    }

def compute_length_ratio(generated: str, reference: str) -> float:
    """Ratio of generated to reference length"""
    gen_len = len(generated.split())
    ref_len = len(reference.split())
    return gen_len / ref_len if ref_len > 0 else 0

print("✓ Metrics configured")

✓ Metrics configured


## Step 8: Run Full Pipeline

In [9]:
print("\n" + "="*120)
print("FULL RAG PIPELINE EVALUATION")
print("Testing on all 50 queries from DelucionQA")
print("="*120 + "\n")

results = []
unique_queries = dataset.drop_duplicates(subset=['id']).reset_index(drop=True)

print(f"Processing {len(unique_queries)} queries...\n")

for idx, row in unique_queries.iterrows():
    question = row['question']
    ground_truth = row['response']
    
    try:
        generated = generate_answer(question)
        
        rouge = compute_rouge(generated, ground_truth)
        length_ratio = compute_length_ratio(generated, ground_truth)
        
        results.append({
            'query_id': row['id'],
            'question': question,
            'generated': generated,
            'ground_truth': ground_truth,
            'rouge1': rouge['rouge1'],
            'rougeL': rouge['rougeL'],
            'length_ratio': length_ratio,
        })
        
        print(f"  [{idx+1:2d}/{len(unique_queries)}] ✓ ROUGE-1: {rouge['rouge1']:.3f}, ROUGE-L: {rouge['rougeL']:.3f}")
    except Exception as e:
        print(f"  [{idx+1:2d}/{len(unique_queries)}] ✗ Error: {str(e)[:50]}")

print(f"\n✓ Processed {len(results)}/{len(unique_queries)} queries")


FULL RAG PIPELINE EVALUATION
Testing on all 50 queries from DelucionQA

Processing 50 queries...

  [ 1/50] ✓ ROUGE-1: 0.973, ROUGE-L: 0.973
  [ 2/50] ✓ ROUGE-1: 0.168, ROUGE-L: 0.137
  [ 3/50] ✓ ROUGE-1: 1.000, ROUGE-L: 1.000
  [ 4/50] ✓ ROUGE-1: 0.756, ROUGE-L: 0.600
  [ 5/50] ✓ ROUGE-1: 0.438, ROUGE-L: 0.438
  [ 6/50] ✓ ROUGE-1: 0.576, ROUGE-L: 0.531
  [ 7/50] ✓ ROUGE-1: 0.855, ROUGE-L: 0.816
  [ 8/50] ✓ ROUGE-1: 0.288, ROUGE-L: 0.216
  [ 9/50] ✓ ROUGE-1: 1.000, ROUGE-L: 1.000
  [10/50] ✓ ROUGE-1: 0.375, ROUGE-L: 0.375
  [11/50] ✓ ROUGE-1: 0.355, ROUGE-L: 0.355
  [12/50] ✓ ROUGE-1: 0.400, ROUGE-L: 0.333
  [13/50] ✓ ROUGE-1: 0.254, ROUGE-L: 0.127
  [14/50] ✓ ROUGE-1: 0.158, ROUGE-L: 0.158
  [15/50] ✓ ROUGE-1: 0.517, ROUGE-L: 0.350
  [16/50] ✓ ROUGE-1: 0.747, ROUGE-L: 0.735
  [17/50] ✓ ROUGE-1: 0.643, ROUGE-L: 0.476
  [18/50] ✓ ROUGE-1: 0.649, ROUGE-L: 0.493
  [19/50] ✓ ROUGE-1: 0.921, ROUGE-L: 0.889
  [20/50] ✓ ROUGE-1: 0.653, ROUGE-L: 0.653
  [21/50] ✓ ROUGE-1: 0.809, ROUGE-L: 0.80

## Step 9: Results Summary

In [10]:
if results:
    df_results = pd.DataFrame(results)
    
    print("\n" + "="*120)
    print("FULL RESULTS SUMMARY")
    print("="*120)
    
    # Statistics
    rouge1_mean = df_results['rouge1'].mean()
    rouge1_std = df_results['rouge1'].std()
    rougeL_mean = df_results['rougeL'].mean()
    rougeL_std = df_results['rougeL'].std()
    length_mean = df_results['length_ratio'].mean()
    length_std = df_results['length_ratio'].std()
    
    print(f"\n📊 ROUGE Scores:")
    print(f"  ROUGE-1: {rouge1_mean:.4f} ± {rouge1_std:.4f}")
    print(f"  ROUGE-L: {rougeL_mean:.4f} ± {rougeL_std:.4f}")
    print(f"\n📊 Length Analysis:")
    print(f"  Avg Length Ratio (Gen/Ref): {length_mean:.3f} ± {length_std:.3f}")
    print(f"  (1.0 = same length as ground truth)")
    
    print(f"\n📊 Score Distribution:")
    print(f"  ROUGE-1:")
    print(f"    Min: {df_results['rouge1'].min():.4f}")
    print(f"    Q1:  {df_results['rouge1'].quantile(0.25):.4f}")
    print(f"    Med: {df_results['rouge1'].median():.4f}")
    print(f"    Q3:  {df_results['rouge1'].quantile(0.75):.4f}")
    print(f"    Max: {df_results['rouge1'].max():.4f}")
    
    print(f"\n  ROUGE-L:")
    print(f"    Min: {df_results['rougeL'].min():.4f}")
    print(f"    Q1:  {df_results['rougeL'].quantile(0.25):.4f}")
    print(f"    Med: {df_results['rougeL'].median():.4f}")
    print(f"    Q3:  {df_results['rougeL'].quantile(0.75):.4f}")
    print(f"    Max: {df_results['rougeL'].max():.4f}")
    
    # Top performing
    print(f"\n🏆 Top 5 Performing Queries:")
    top5 = df_results.nlargest(5, 'rouge1')[['query_id', 'rouge1', 'rougeL']]
    for idx, row in top5.iterrows():
        print(f"  [{row['query_id']:3d}] ROUGE-1: {row['rouge1']:.4f}, ROUGE-L: {row['rougeL']:.4f}")
    
    # Bottom performing
    print(f"\n⚠️  Bottom 5 Performing Queries:")
    bottom5 = df_results.nsmallest(5, 'rouge1')[['query_id', 'rouge1', 'rougeL']]
    for idx, row in bottom5.iterrows():
        print(f"  [{row['query_id']:3d}] ROUGE-1: {row['rouge1']:.4f}, ROUGE-L: {row['rougeL']:.4f}")
    
    print("\n" + "="*120)
    
    # Full results table
    print("\nDetailed Results:")
    display(df_results[['query_id', 'rouge1', 'rougeL', 'length_ratio']])
else:
    print("No results generated")


FULL RESULTS SUMMARY

📊 ROUGE Scores:
  ROUGE-1: 0.6133 ± 0.2324
  ROUGE-L: 0.5461 ± 0.2505

📊 Length Analysis:
  Avg Length Ratio (Gen/Ref): 1.776 ± 1.410
  (1.0 = same length as ground truth)

📊 Score Distribution:
  ROUGE-1:
    Min: 0.1579
    Q1:  0.4325
    Med: 0.6461
    Q3:  0.7989
    Max: 1.0000

  ROUGE-L:
    Min: 0.1270
    Q1:  0.3601
    Med: 0.5035
    Q3:  0.7644
    Max: 1.0000

🏆 Top 5 Performing Queries:


ValueError: Unknown format code 'd' for object of type 'str'

## Step 10: Save Results

In [ ]:
if results:
    # Save full results
    df_results.to_csv('/Users/saikrishna/Desktop/Codespace/IIIT_AI_ML_course/Capstone_Project/reliablerag/delucion_dataset/rag_results.csv', index=False)
    print("✓ Saved results to rag_results.csv")
    
    # Save summary
    summary = {
        'total_queries': len(df_results),
        'rouge1_mean': float(rouge1_mean),
        'rouge1_std': float(rouge1_std),
        'rougeL_mean': float(rougeL_mean),
        'rougeL_std': float(rougeL_std),
        'length_ratio_mean': float(length_mean),
        'length_ratio_std': float(length_std),
        'rouge1_min': float(df_results['rouge1'].min()),
        'rouge1_max': float(df_results['rouge1'].max()),
        'rougeL_min': float(df_results['rougeL'].min()),
        'rougeL_max': float(df_results['rougeL'].max()),
    }
    
    with open('/Users/saikrishna/Desktop/Codespace/IIIT_AI_ML_course/Capstone_Project/reliablerag/delucion_dataset/rag_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print("✓ Saved summary to rag_summary.json")